<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/slm_llamaindex_practical_simple_with_outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Support-Desk Assistant — SLMs + LlamaIndex

**A practical for the session "Small Language Models & LlamaIndex".**

You will build a question-answering assistant over a small set of company
policy documents. Everything runs on **free, openly available models** — no API key,
no paid service, and it works on a plain Colab CPU runtime (a T4 GPU just makes it faster).

| Role | Model | Size |
|---|---|---|
| Generation (the SLM) | `Qwen/Qwen2.5-0.5B-Instruct` | 0.5B |
| Embeddings (search) | `BAAI/bge-small-en-v1.5` | 33M |
| Reranking | `cross-encoder/ms-marco-MiniLM-L-6-v2` | 22M |

**How to use this notebook**

1. Run every cell top to bottom, once, without changing anything.
2. Then go back to the three cells marked **🧪 EXPERIMENT** — that is where the learning happens.
3. Not a programmer? Read the text, run the cells, and look at the printed output.
   Every code cell starts with a comment box saying **what it does, its input and its output**.

**The big picture (RAG = Retrieval-Augmented Generation)**

`Documents → cut into chunks → index (search by meaning) → find best chunks for a question → small model writes the answer from those chunks`

> **Tip:** in Colab choose *Runtime → Change runtime type → T4 GPU* if it is offered.
> On CPU everything still works; each answer just takes a few seconds.

### 📖 Words you will see (plain-English glossary)

| Word | What it means in everyday terms |
|---|---|
| **Model** | A computer program that has learned from lots of text. Think of it as a very well-read assistant. |
| **SLM (Small Language Model)** | A *small* model. Less knowledgeable than the giants (like ChatGPT) but fast, free and runs on a normal laptop. |
| **Document** | One whole file (for us: one policy). |
| **Chunk / Node** | A short piece of a document - like an index card cut from a book. |
| **Embedding** | A list of numbers that captures the *meaning* of a piece of text, so the computer can tell that "money back" and "refund" mean the same thing. |
| **Index** | A catalogue of all chunks, organised by meaning so they can be found quickly. |
| **Retrieval** | Finding the chunks most relevant to a question - the "search" step. |
| **Reranking** | A second, more careful check that re-orders the search results. |
| **Metadata** | Labels attached to a document (its file name, its category) - like folder tabs. |
| **Token** | The small pieces text is broken into for the model - roughly three-quarters of a word. |
| **Prompt** | The instructions + question we hand to the model. |
| **RAG** | Retrieval-Augmented Generation: *look up* the facts first, *then* let the model write the answer from them. |

**Reading the code:** lines starting with `#` are plain-English notes for you - the computer ignores them.
You never need to change any code unless an 🧪 EXPERIMENT tells you to.

---
## Step 1 — Install

This installs LlamaIndex plus the Hugging Face tools. It takes 2–3 minutes the
first time. Run it once per session. If Colab asks you to restart the runtime, do it,
then continue from Step 2.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Like installing apps on a phone - we download the tools this notebook needs. Do it once.
# WHAT THIS CELL DOES : Installs the Python libraries (packages) we need.
# INPUT               : Nothing - it downloads packages from the internet.
# OUTPUT              : Nothing visible if it works (the "-q" flag keeps it quiet).
# =============================================================================
# "%pip install" is the notebook version of the normal "pip install" command.
%pip install -q \
    "llama-index-core>=0.14,<0.15" \
    "llama-index-llms-huggingface>=0.6" \
    "llama-index-embeddings-huggingface>=0.6" \
    "llama-index-retrievers-bm25>=0.7" \
    "transformers>=4.44" accelerate "sentence-transformers>=3.0" \
    bm25s PyStemmer

# What each package is for:
#   llama-index-core                    -> LlamaIndex itself: connects documents, search and the model
#   llama-index-llms-huggingface        -> lets LlamaIndex use a Hugging Face language model (our SLM)
#   llama-index-embeddings-huggingface  -> lets LlamaIndex use a Hugging Face embedding model (search by meaning)
#   llama-index-retrievers-bm25         -> keyword search (BM25), used in Step 8
#   transformers                        -> Hugging Face library that downloads and runs the models
#   accelerate                          -> puts the model on the GPU automatically when there is one
#   sentence-transformers               -> runs the embedding model and the reranker model
#   bm25s, PyStemmer                    -> helpers for keyword search (PyStemmer: "refunds" -> "refund")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\satis\AppData\Local\Temp\claude\E--Projects-SODApp\e58991e7-7d27-43f0-84fd-8bd80a2ac316\scratchpad\venv\Scripts\python.exe -m pip install --upgrade pip


In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Checks whether this computer has a fast graphics chip (GPU). No GPU? No problem - just a bit slower .
# WHAT THIS CELL DOES : Checks whether a GPU is available and hides noisy warnings.
# INPUT               : Nothing.
# OUTPUT              : "Running on: cuda" (GPU, fast)  or  "Running on: cpu" (slower, still works).
# =============================================================================
import warnings   # WHY: built-in Python tool; lets us hide harmless warning messages
import torch      # WHY: PyTorch is the engine that runs the models' maths (on CPU or GPU)

warnings.filterwarnings("ignore")   # hide harmless warnings so the output stays readable

# Use the GPU ("cuda") if there is one, otherwise the CPU. Later cells reuse DEVICE.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

Running on: cpu


---
## Step 2 — Configure the models

`Settings` is LlamaIndex's global configuration. Whatever you put here is used by
every component you build afterwards.

**This is the most important cell in the notebook.** If you skip
`Settings.embed_model`, LlamaIndex silently falls back to OpenAI and later fails with
*"no API key found"*.

Two models are loaded:

* **the embedding model** — turns text into a list of 384 numbers ("coordinates of meaning")
  so we can search by meaning rather than by exact words;
* **the small language model (SLM)** — reads the passages we retrieve and writes the answer.

`max_new_tokens=256` caps the answer length. `do_sample=False` makes the model always pick
its most likely next word, so runs are repeatable and it invents less — what you want for
factual Q&A.

The first run downloads roughly 1 GB of model files. Be patient; they are cached afterwards.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Downloads the two 'brains' we use: one that understands meaning, one that writes answers.
# WHAT THIS CELL DOES : Loads the two models and registers them as LlamaIndex's defaults.
#                       (1) Embedding model - converts text into numbers so we can search by meaning.
#                       (2) Small Language Model - reads the retrieved text and writes the answer.
# INPUT               : The two model names below. (~1 GB downloads on the first run.)
# OUTPUT              : "Embedding dimensions: 384" - each piece of text becomes a list of 384 numbers.
# =============================================================================
from llama_index.core import Settings                                # WHY: LlamaIndex's global "control panel"
from llama_index.embeddings.huggingface import HuggingFaceEmbedding  # WHY: loads a Hugging Face embedding model
from llama_index.llms.huggingface import HuggingFaceLLM              # WHY: loads a Hugging Face language model

EMBED_MODEL = "BAAI/bge-small-en-v1.5"       # small embedding model (33 million parameters)
LLM_MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"   # small language model (0.5 billion parameters)

# IMPORTANT: register the embedding model. Without it LlamaIndex tries OpenAI and asks for an API key.
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL, device=DEVICE)

# Register the language model (the SLM).
Settings.llm = HuggingFaceLLM(
    model_name=LLM_MODEL,
    tokenizer_name=LLM_MODEL,              # the tokenizer splits text into small pieces (tokens) the model reads
    context_window=2048,                   # model's "working memory" in tokens: documents + question + answer must fit
    max_new_tokens=256,                    # maximum length of one answer
    generate_kwargs={"do_sample": False},  # False = always pick the most likely next word -> same answer every run
    is_chat_model=True,                    # format prompts the way a chat model expects
    device_map="auto",                     # GPU if available, otherwise CPU
)

# Quick test: turn the word "hello" into numbers and count them.
print("Embedding dimensions:", len(Settings.embed_model.get_text_embedding("hello")))   # -> 384

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2844.31it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:  10%|█         | 29/290 [00:00<00:00, 274.53it/s]

Loading weights:  20%|█▉        | 57/290 [00:00<00:01, 203.03it/s]

...

Loading weights:  38%|███▊      | 109/290 [00:00<00:01, 101.86it/s]

Loading weights:  42%|████▏     | 121/290 [00:00<00:01, 101.94it/s]

Loading weights:  46%|████▌     | 132/290 [00:01<00:01, 98.58it/s] 

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 244.85it/s]

Embedding dimensions: 384


### A quick sanity check

Ask the model something it cannot possibly know. It has never seen our company's
policies, so whatever it says is invented.

**This is the problem RAG solves.** We ask the same question again at the end of Step 10,
this time with retrieval switched on.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Shows what happens WITHOUT our documents: the model guesses. This is the problem we are about to fix.
# WHAT THIS CELL DOES : Asks the model a question WITHOUT giving it any documents.
# INPUT               : A question about "Northwind Supply" (a made-up company).
# OUTPUT              : A vague or invented answer - the model does not know the real policy.
# =============================================================================
# .complete() sends text to the model and returns its reply.
print(Settings.llm.complete("How long does Northwind Supply take to process a refund?"))

 I'm sorry, but I cannot provide an answer to your question as there is no specific information available about how long Northwind Supply takes to process refunds. The processing time for refunds can vary depending on the company's policies and procedures, as well as any external factors such as holidays or weekends. It is best to contact Northwind Supply directly to inquire about their refund processing times. Alternatively, you may want to check their website or customer support resources for more up-to-date information.


---
## Step 3 — Create and load the documents

A real project would point at a folder of PDFs or Word files. To keep this
self-contained we write six short policy documents to a `data` folder, then load them
with `SimpleDirectoryReader`.

While loading we attach **metadata** (a *category* and a *source* file name) to every
document. Metadata is cheap to add now and hard to add later — you cannot filter or
cite on a field you never stored.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Creates six small company policy files - our pretend company's rulebook.
# WHAT THIS CELL DOES : Creates six sample policy files and saves them in a folder called "data".
# INPUT               : The POLICIES dictionary below  { "file name": "file text" }.
#                       You can edit the text or add your own policy here.
# OUTPUT              : Six files in ./data (refunds.md, shipping.md, ...) and a message listing them.
# =============================================================================
import os   # WHY: built-in Python tool for working with files and folders

os.makedirs("data", exist_ok=True)   # create the "data" folder (does nothing if it already exists)

POLICIES = {
    "refunds.md": """# Refund Policy

## Eligibility
Customers may request a refund within 30 days of the original purchase date.
Proof of purchase is required, and the order number must be quoted on the request.
Items returned in a condition that prevents resale may be refused or refunded in part.
Digital gift cards are non-refundable once the redemption code has been revealed.

## Processing times
Approved refunds are issued to the original payment method within 5 business days.
For international orders, refunds take 10 business days because of cross-border
settlement times between the acquiring and issuing banks.
If the original payment method has been closed, we issue store credit instead, and
that credit appears on the account within 2 business days.

## Subscriptions
Annual subscription plans are refunded on a pro-rata basis for the unused months.
Monthly plans are not refunded for the current period, but they can be cancelled so
that no further charge is taken.
Refunds on subscriptions purchased through a reseller must be requested from that reseller.
""",
    "shipping.md": """# Shipping Policy

## Domestic delivery
Standard delivery arrives in 3 to 5 business days within the country.
Express delivery arrives the next business day for orders placed before 2:00 pm.
Orders placed after 2:00 pm on a Friday are dispatched on the following Monday.
Orders above 5,000 in value ship free of charge on the standard service.

## International delivery
International shipping takes 7 to 14 business days and may be delayed by customs.
Import duties and local taxes are the responsibility of the recipient.
We do not currently ship hardware to addresses in countries under trade restriction.

## Tracking and problems
A tracking number is emailed within 24 hours of dispatch.
If tracking has not updated for 5 business days, contact support and we will open a
carrier investigation, which normally concludes within 10 business days.
""",
    "warranty.md": """# Warranty Policy

## What is covered
All hardware carries a 24-month manufacturer warranty from the date of delivery.
The warranty covers manufacturing defects in materials and workmanship.
Accessories supplied in the box carry a 12-month warranty.

## What is not covered
The warranty does not cover accidental damage, liquid damage, damage caused by
unauthorised repair, or normal wear of consumable parts such as batteries and cables.
Cosmetic scratches that do not affect function are not covered.

## Making a claim
A warranty claim requires an RMA number, which support issues within one business day.
The RMA number must be written on the outside of the returned package.
Replacement units are dispatched within 7 business days of the faulty unit being received.
If the same unit fails twice within the warranty period we replace it with a new unit
rather than a repaired one.
""",
    "accounts.md": """# Account Access Policy

## Passwords
You can reset your password from the Settings page of your account.
A password reset link expires 60 minutes after it is issued.
Passwords must be at least 12 characters and cannot repeat any of your last 5 passwords.

## Two-factor authentication
Two-factor authentication can be enabled from Settings and is mandatory for admin users.
Recovery codes are shown once at setup and cannot be retrieved afterwards.
If you lose both your second factor and your recovery codes, identity verification with
support takes up to 2 business days.

## Lockouts
An account is locked after 5 consecutive failed sign-in attempts.
A locked account unlocks automatically after 30 minutes, or immediately if support
verifies your identity.
Repeated lockouts within 24 hours trigger a manual security review.
""",
    "privacy.md": """# Privacy Policy

## Where data is held
Customer data is stored in data centres located in the European Union.
Backups are encrypted at rest and are also held within the European Union.
We do not sell customer data to third parties under any circumstances.

## Retention
Account records are retained for 24 months after an account is closed.
Support conversations are retained for 12 months for quality and training purposes.
Billing records are retained for 7 years because tax law requires it, even after deletion
of the rest of the account.

## Your rights
You may request deletion of your personal data at any time; deletion completes within 30 days.
You may request an export of your data, which is delivered as a JSON archive within 14 days.
Requests are made from the Privacy tab in Settings, or by writing to the data protection team.
""",
    "business.md": """# Business Accounts Policy

## Eligibility and pricing
Business accounts are available to organisations with 5 or more seats.
A volume discount of 15 percent applies above 50 seats, and 25 percent above 200 seats.
Educational and non-profit organisations receive an additional 10 percent on top of the
volume discount, subject to verification.

## Billing
Business invoices are issued monthly on net-30 payment terms.
Purchase order numbers can be attached to invoices from the Billing tab.
Late payment beyond 45 days suspends new seat provisioning until the balance is cleared.

## Account management
Organisations above 200 seats are assigned a dedicated account manager.
Seat counts can be adjusted once per billing cycle, and reductions take effect at the
start of the next cycle rather than immediately.
Single sign-on is available on business plans at no extra cost.
""",
}

# Save each policy as its own file inside the "data" folder.
for file_name, text in POLICIES.items():
    with open(os.path.join("data", file_name), "w") as f:   # "w" = write (creates the file)
        f.write(text)

print("Wrote", len(POLICIES), "policy files to ./data:", ", ".join(POLICIES))

Wrote 6 policy files to ./data: refunds.md, shipping.md, warranty.md, accounts.md, privacy.md, business.md


In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Opens those six files and sticks a label on each one (like a folder tab: 'billing', 'shipping'...).
# WHAT THIS CELL DOES : Reads the files from "data" into LlamaIndex "Documents" and tags each one
#                       with metadata: its source file name and a category.
# INPUT               : The six files in ./data.
# OUTPUT              : "Loaded 6 documents" and one line per document: file name, category, length.
#                       (A warning about "llama-index-readers-file" can be ignored.)
# =============================================================================
from llama_index.core import SimpleDirectoryReader   # WHY: reads every file in a folder into Documents

# Which category each file belongs to. Step 7 uses this to filter search results.
CATEGORY = {
    "refunds.md":  "billing",
    "shipping.md": "shipping",
    "warranty.md": "warranty",
    "accounts.md": "account",
    "privacy.md":  "privacy",
    "business.md": "billing",
}

# Called once per file; returns the labels (metadata) to attach to it.
#   input : "data/refunds.md"
#   output: {"source": "refunds.md", "category": "billing"}
def file_metadata(path):
    name = os.path.basename(path)   # "data/refunds.md" -> "refunds.md"
    return {"source": name, "category": CATEGORY[name]}

# Load every file in ./data and attach the metadata from the function above.
docs = SimpleDirectoryReader("data", file_metadata=file_metadata).load_data()

print(f"Loaded {len(docs)} documents\n")
for d in docs:
    print(f"  {d.metadata['source']:<14} category={d.metadata['category']:<9} {len(d.text)} characters")

Loaded 6 documents

  accounts.md    category=account   847 characters
  business.md    category=billing   892 characters
  privacy.md     category=privacy   860 characters
  refunds.md     category=billing   1074 characters
  shipping.md    category=shipping  867 characters
  warranty.md    category=warranty  903 characters


---
## Step 4 — Chunk into Nodes

A **Document** is a whole file. A **Node** is a *chunk* of that file — the unit that
actually gets embedded and retrieved.

Why split at all?

1. A chunk about one idea gives a sharp, specific "meaning point". A chunk about eight
   ideas gives a blurry average that matches nothing well.
2. Our model has a 2048-token window. Everything we retrieve must fit in it, together
   with the question and the answer.

`chunk_overlap` repeats a little text across each boundary so an answer that straddles
two chunks is still whole in at least one of them.

> **Why 128 tokens?** Our documents are tiny (~220 tokens each). With 512 every document
> would stay in one piece and there would be nothing to see. 128 gives section-sized chunks,
> like a real multi-page policy would. Always measure chunk size on *your* documents.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Cuts each long document into short paragraphs, like cutting a book into index cards.
# WHAT THIS CELL DOES : Cuts each document into small pieces called "nodes" (chunks).
# INPUT               : The 6 documents loaded in Step 3.
# OUTPUT              : "6 documents -> N nodes" and a preview of the first node.
# =============================================================================
from llama_index.core.node_parser import SentenceSplitter   # WHY: cuts text into chunks without breaking sentences

# chunk_size    = maximum chunk length in tokens (1 token is roughly 3/4 of a word)
# chunk_overlap = tokens repeated between neighbouring chunks so no sentence loses its context
splitter = SentenceSplitter(chunk_size=128, chunk_overlap=16)

# Do the splitting. Every node remembers its document's metadata (source, category).
nodes = splitter.get_nodes_from_documents(docs)

print(f"{len(docs)} documents  ->  {len(nodes)} nodes\n")
print("Example node:", nodes[0].metadata)
print(nodes[0].text[:300], "...")

6 documents  ->  12 nodes

Example node: {'source': 'accounts.md', 'category': 'account'}
# Account Access Policy

## Passwords
You can reset your password from the Settings page of your account.
A password reset link expires 60 minutes after it is issued.
Passwords must be at least 12 characters and cannot repeat any of your last 5 passwords.

## Two-factor authentication
Two-fa ...


### 🧪 EXPERIMENT 1 — chunk size

The cell below splits the same six documents at four chunk sizes so you can see the trade-off.

* Small chunks → many nodes, each very specific. Precise, but an answer spanning several
  sentences can get cut in half.
* Large chunks → few nodes, each covering several topics. Retrieval gets vague, and each
  chunk uses more of the model's limited context window.

Then change `chunk_size` in the cell *above*, re-run Step 4 onwards, and watch how the
retrieval scores in Step 6 move.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Tries different 'index card' sizes so you can see how the size changes the number of cards.
# WHAT THIS CELL DOES : Splits the same documents at four different chunk sizes.
# INPUT               : The documents, and the chunk sizes to try: 96, 192, 384, 1024.
# OUTPUT              : One line per size: number of nodes and their average length in characters.
#                       Smaller chunk size -> more, shorter nodes.
# =============================================================================
for size in (96, 192, 384, 1024):
    # overlap = 12% of the chunk size (int() rounds it to a whole number)
    pieces = SentenceSplitter(chunk_size=size, chunk_overlap=int(size * 0.12)).get_nodes_from_documents(docs)
    average = sum(len(p.text) for p in pieces) / len(pieces)   # average characters per node
    print(f"chunk_size={size:<5} -> {len(pieces):>3} nodes, average {average:>6.0f} characters each")

chunk_size=96    ->  15 nodes, average    360 characters each
chunk_size=192   ->   7 nodes, average    790 characters each
chunk_size=384   ->   6 nodes, average    905 characters each
chunk_size=1024  ->   6 nodes, average    905 characters each


---
## Step 5 — Build the index

`VectorStoreIndex` turns every node into an embedding (a list of 384 numbers) and stores
them so they can be searched by meaning. With only a dozen nodes this takes a few seconds.

> If you change the chunk size or the embedding model, **re-run this cell** so the
> index is rebuilt from the new nodes.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Builds a smart catalogue of all the cards so we can find them by meaning (like a librarian who understands topics).
# WHAT THIS CELL DOES : Builds the search index. Every node is converted to numbers (an embedding)
#                       and stored so we can search by meaning.
# INPUT               : The nodes from Step 4.
# OUTPUT              : A short confirmation message.
# =============================================================================
from llama_index.core import VectorStoreIndex   # WHY: the index that stores embeddings and searches them

index = VectorStoreIndex(nodes)   # embeds every node using the embedding model from Step 2
print("Index built from", len(nodes), "nodes.")

Index built from 12 nodes.


---
## Step 6 — Retrieval: the part that decides everything

`similarity_top_k` is how many chunks we fetch. The score is *cosine similarity* —
how close the question and the chunk are in meaning (higher = closer).

Notice that *"How long do refunds take?"* matches a passage that never says "how long".
That is semantic search doing its job.

**Build this habit:** whenever an answer is wrong, print the retrieved chunks *first*.
If the right passage is not in this list, no prompt and no model can save the answer.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Looks up the cards that best match a question - this is the 'search' part, before any answer is written.
# WHAT THIS CELL DOES : Searches the index for the chunks that best match a question ("retrieval").
# INPUT               : Two example questions.
# OUTPUT              : For each question, the 4 best chunks with score (higher = closer in meaning),
#                       source file and category.
# =============================================================================
# Helper used in many later cells: prints search results in a readable way.
#   input : a list of search results ("hits") and an optional title
#   output: one numbered line per result with score, file, category and the first 150 characters
def show(hits, title=""):
    if title:
        print(title)
        print("=" * 78)
    for i, hit in enumerate(hits, 1):
        text = " ".join(hit.text.split())   # collapse line breaks and extra spaces into one line
        print(f"{i}. score={hit.score:.3f}  [{hit.metadata['source']} / {hit.metadata['category']}]")
        print(f"   {text[:150]}...")
    print()

# A "retriever" is the search tool. similarity_top_k=4 -> return the 4 best matches.
retriever = index.as_retriever(similarity_top_k=4)

show(retriever.retrieve("How long do refunds take?"), "QUERY: How long do refunds take?")
show(retriever.retrieve("I am locked out of my account"), "QUERY: I am locked out of my account")

QUERY: How long do refunds take?
1. score=0.793  [refunds.md / billing]
   # Refund Policy ## Eligibility Customers may request a refund within 30 days of the original purchase date. Proof of purchase is required, and the ord...
2. score=0.753  [refunds.md / billing]
   If the original payment method has been closed, we issue store credit instead, and that credit appears on the account within 2 business days. ## Subsc...
3. score=0.667  [warranty.md / warranty]
   Cosmetic scratches that do not affect function are not covered. ## Making a claim A warranty claim requires an RMA number, which support issues within...
4. score=0.642  [privacy.md / privacy]
   ## Your rights You may request deletion of your personal data at any time; deletion completes within 30 days. You may request an export of your data, ...

QUERY: I am locked out of my account
1. score=0.748  [accounts.md / account]
   ## Lockouts An account is locked after 5 consecutive failed sign-in attempts. A locked account unloc

---
## Step 7 — Metadata filters

A filter narrows the search *before* it happens. Two benefits:

* **Precision** — plausible-but-wrong matches from other categories disappear.
* **Access control** — in a real system you filter by what the current user may see,
  so the model physically cannot read a document it should not. That is far stronger
  than telling a model to keep a secret.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Searches again but only inside one folder (category), to show how labels make search more accurate.
# WHAT THIS CELL DOES : Runs the same search twice - once normally, once limited to one category.
# INPUT               : The vague question "How long does it take?" and the filter category = "billing".
# OUTPUT              : Two result lists. Without the filter, unrelated topics may appear.
#                       With the filter, only billing documents (refunds.md, business.md) can appear.
# =============================================================================
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter   # WHY: build a search filter

# The filter reads: "only keep chunks whose category equals 'billing'".
billing_only = MetadataFilters(filters=[MetadataFilter(key="category", value="billing")])

question = "How long does it take?"

show(index.as_retriever(similarity_top_k=3).retrieve(question),
     f"QUERY: {question}   (no filter)")

show(index.as_retriever(similarity_top_k=3, filters=billing_only).retrieve(question),
     f"QUERY: {question}   (filtered to category='billing')")
#In a real app, something else chooses the filter. That could be a dropdown the user picks, the user’s access rights, or an extra model call that classifies the question first

QUERY: How long does it take?   (no filter)
1. score=0.645  [privacy.md / privacy]
   ## Your rights You may request deletion of your personal data at any time; deletion completes within 30 days. You may request an export of your data, ...
2. score=0.617  [accounts.md / account]
   # Account Access Policy ## Passwords You can reset your password from the Settings page of your account. A password reset link expires 60 minutes afte...
3. score=0.606  [shipping.md / shipping]
   Import duties and local taxes are the responsibility of the recipient. We do not currently ship hardware to addresses in countries under trade restric...

QUERY: How long does it take?   (filtered to category='billing')
1. score=0.588  [refunds.md / billing]
   # Refund Policy ## Eligibility Customers may request a refund within 30 days of the original purchase date. Proof of purchase is required, and the ord...
2. score=0.558  [refunds.md / billing]
   If the original payment method has been closed, we issue stor

---
## Step 8 — Hybrid retrieval (meaning + keywords)

Embeddings are great at paraphrase but weak on exact tokens. An RMA number (Return Merchandise Authorization, a reference number support issues when you claim a warranty) or an
error code has no useful "neighbourhood" in meaning-space.

**BM25** keyword search is the opposite: unbeatable on exact words, blind to paraphrase.

`QueryFusionRetriever` runs both and merges the two ranked lists, so you get both strengths.
`num_queries=1` skips LLM-written query variations, which would add slow extra model calls.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Combines two ways of searching: by meaning AND by exact words, to catch things like code numbers.
# WHAT THIS CELL DOES : Combines two kinds of search ("hybrid"):
#                       - vector search : finds text with similar MEANING
#                       - BM25 search   : finds text containing the exact KEYWORDS
# INPUT               : The keyword-style query "RMA number".
# OUTPUT              : Top 3 results from vector search alone, then top 3 from hybrid search.
#                       Hybrid scores look tiny (about 0.03) - that is normal: they are rank-based
#                       and cannot be compared with the vector scores above.
# =============================================================================
from llama_index.retrievers.bm25 import BM25Retriever          # WHY: keyword search
from llama_index.core.retrievers import QueryFusionRetriever   # WHY: runs several searches and merges the results

# Keyword retriever, built directly from our nodes.
bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=5)

# Hybrid retriever = vector search + keyword search.
hybrid = QueryFusionRetriever(
    [index.as_retriever(similarity_top_k=5), bm25],
    similarity_top_k=5,        # number of merged results to return
    num_queries=1,             # 1 = only the original question (no extra LLM-written variations -> faster)
    mode="reciprocal_rerank",  # merge rule: chunks ranked high by BOTH searches rise to the top
    use_async=False,           # run searches one after another (avoids problems inside notebooks)
)

question = "RMA number"
show(index.as_retriever(similarity_top_k=3).retrieve(question), f"QUERY: {question}   (vector only)")
show(hybrid.retrieve(question)[:3],                            f"QUERY: {question}   (hybrid: vector + BM25)")

QUERY: RMA number   (vector only)
1. score=0.680  [warranty.md / warranty]
   Cosmetic scratches that do not affect function are not covered. ## Making a claim A warranty claim requires an RMA number, which support issues within...
2. score=0.619  [shipping.md / shipping]
   Import duties and local taxes are the responsibility of the recipient. We do not currently ship hardware to addresses in countries under trade restric...
3. score=0.612  [warranty.md / warranty]
   # Warranty Policy ## What is covered All hardware carries a 24-month manufacturer warranty from the date of delivery. The warranty covers manufacturin...

QUERY: RMA number   (hybrid: vector + BM25)
1. score=0.033  [warranty.md / warranty]
   Cosmetic scratches that do not affect function are not covered. ## Making a claim A warranty claim requires an RMA number, which support issues within...
2. score=0.033  [shipping.md / shipping]
   Import duties and local taxes are the responsibility of the recipient. We do not curr

---
## Step 9 — Reranking: retrieve wide, keep narrow

The embedding search compares two vectors that were computed **separately** — fast but approximate.

A **cross-encoder** reads the question and the passage *together* and scores the pair directly.
Much more accurate, but far too slow to run over a whole collection. So we use the cheap search to
get 10 candidates, then let the cross-encoder pick the best 3.

This matters even more for a small model: we can only afford about three chunks, so they had
better be the right ones.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : A second, more careful judge re-reads the top results and puts the truly best ones first.
# WHAT THIS CELL DOES : Re-orders search results using a "reranker" - a more careful model that reads
#                       the question and each chunk together and scores how well they match.
# INPUT               : The question "Can I get money back on a yearly plan?" and 10 candidate chunks.
# OUTPUT              : First the top 5 of the 10 candidates in their original order, then the best 3
#                       after reranking. Reranker scores can be negative - only the ORDER matters
#                       (higher = better match).
# =============================================================================
from llama_index.core.postprocessor import SentenceTransformerRerank   # WHY: the reranker
from llama_index.core import QueryBundle                               # WHY: wraps the question so the reranker can read it

# Load the reranker. top_n=3 -> keep only the best 3 after re-scoring.
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=3,
    device=DEVICE,
)

question = "Can I get money back on a yearly plan?"

# A: quick, rough search - fetch a wide set of 10 candidates.
wide = index.as_retriever(similarity_top_k=10).retrieve(question)
show(wide[:5], f"QUERY: {question}   (top 5 of 10 before reranking)")

# B: careful re-scoring - keep only the best 3.
show(reranker.postprocess_nodes(wide, QueryBundle(question)), f"QUERY: {question}   (after reranking)")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4827.17it/s]

QUERY: Can I get money back on a yearly plan?   (top 5 of 10 before reranking)
1. score=0.727  [refunds.md / billing]
   If the original payment method has been closed, we issue store credit instead, and that credit appears on the account within 2 business days. ## Subsc...
2. score=0.652  [refunds.md / billing]
   # Refund Policy ## Eligibility Customers may request a refund within 30 days of the original purchase date. Proof of purchase is required, and the ord...
3. score=0.606  [business.md / billing]
   # Business Accounts Policy ## Eligibility and pricing Business accounts are available to organisations with 5 or more seats. A volume discount of 15 p...
4. score=0.585  [privacy.md / privacy]
   # Privacy Policy ## Where data is held Customer data is stored in data centres located in the European Union. Backups are encrypted at rest and are al...
5. score=0.574  [warranty.md / warranty]
   Cosmetic scratches that do not affect function are not covered. ## Making a claim A warranty

QUERY: Can I get money back on a yearly plan?   (after reranking)
1. score=-1.523  [refunds.md / billing]
   If the original payment method has been closed, we issue store credit instead, and that credit appears on the account within 2 business days. ## Subsc...
2. score=-10.414  [refunds.md / billing]
   # Refund Policy ## Eligibility Customers may request a refund within 30 days of the original purchase date. Proof of purchase is required, and the ord...
3. score=-11.112  [business.md / billing]
   # Business Accounts Policy ## Eligibility and pricing Business accounts are available to organisations with 5 or more seats. A volume discount of 15 p...



---
## Step 10 — The query engine: retrieval + a grounded prompt

Now we join everything: retrieve 10 → rerank to 3 → hand those to the small model with a strict prompt.

Two things in the prompt do the heavy lifting:

1. **"Use ONLY the context"** — keeps the model to the retrieved text.
2. **"If the answer is not there, say you do not know"** — gives it a sanctioned way to fail.
   Without an exit, a model with nothing to say will invent something.

Small models follow short, literal instructions better than long, nuanced ones. Keep the prompt boring.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Puts it all together: find cards -> pick the best 3 -> the model writes a short answer using only those.
# WHAT THIS CELL DOES : Builds the complete question-answering system (a "query engine"):
#                       search 10 chunks -> rerank to best 3 -> give them to the small model -> answer.
# INPUT               : Four questions that the documents DO answer.
# OUTPUT              : For each question: the answer (A:) and the source files used, with reranker scores.
#                       Small models can still make mistakes - always compare the answer with the sources.
# =============================================================================
from llama_index.core.prompts import PromptTemplate   # WHY: lets us write our own instructions for the model

# The instructions sent to the model. {context_str} is replaced by the retrieved chunks,
# {query_str} by the user's question.
QA_PROMPT = PromptTemplate(
    "You are a customer support assistant for Northwind Supply.\n"
    "Answer the question using ONLY the context below.\n"                  # keeps the model on our documents
    "If the context does not contain the answer, reply exactly: "          # gives the model a safe way out
    "\"I don't know based on the policy documents.\"\n"
    "Answer in one or two sentences.\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer: "
)

# Assemble the query engine.
engine = index.as_query_engine(
    similarity_top_k=10,             # first fetch 10 candidate chunks
    node_postprocessors=[reranker],  # then rerank and keep the best 3
    text_qa_template=QA_PROMPT,      # use our instructions above
)

# Helper: ask a question and print the answer plus the sources behind it.
#   input : a question (text)
#   output: printed "Q:", "A:" and the source files with their reranker scores
def ask(question):
    response = engine.query(question)
    print("Q:", question)
    print("A:", str(response).strip())
    print("   sources:", ", ".join(f"{n.metadata['source']}({n.score:.2f})" for n in response.source_nodes))
    print("-" * 78)

ask("How long do refunds take?")
ask("What happens if I fail to sign in five times?")
ask("Is accidental damage covered by the warranty?")
ask("What discount applies above 200 seats?")

Q: How long do refunds take?
A: Refunds take 10 business days for international orders.
   sources: refunds.md(6.33), refunds.md(3.46), accounts.md(-4.86)
------------------------------------------------------------------------------


Q: What happens if I fail to sign in five times?
A: If you fail to sign in five times, your account will be locked.
   sources: accounts.md(1.38), warranty.md(-10.12), business.md(-10.95)
------------------------------------------------------------------------------


Q: Is accidental damage covered by the warranty?
A: Accidental damage is not covered by the warranty.
   sources: warranty.md(7.37), warranty.md(-2.91), refunds.md(-11.34)
------------------------------------------------------------------------------


Q: What discount applies above 200 seats?
A: The discount applies above 200 seats, specifically 25% above 200 seats.
   sources: business.md(7.10), business.md(1.55), shipping.md(-11.34)
------------------------------------------------------------------------------


### 🧪 EXPERIMENT 2 — the honesty test

Ask something the documents genuinely do not cover. A well-grounded system should decline;
an ungrounded one invents a confident, plausible, wrong answer.

Run the cell, then **comment out `text_qa_template=QA_PROMPT`** in the cell above (put a `#`
at the start of that line), re-run both cells and compare. It is the clearest demonstration
of why prompt grounding matters *more* for small models, not less.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Asks questions the documents cannot answer, to check the assistant honestly says 'I don't know'.
# WHAT THIS CELL DOES : Tests honesty - two questions the documents do NOT answer, and one they do.
# INPUT               : Three questions; uses the ask() helper from the previous cell.
# OUTPUT              : The first two should say "I don't know based on the policy documents."
#                       The third now gets a real answer from our documents - compare it with the
#                       vague answer the model gave in the Step 2 sanity check.
# =============================================================================
ask("Do you accept payment in cryptocurrency?")
ask("What is the CEO's home address?")
ask("How long does Northwind Supply take to process a refund?")   # same question as the Step 2 sanity check

Q: Do you accept payment in cryptocurrency?
A: No, I don't know based on the policy documents.
   sources: refunds.md(-11.09), refunds.md(-11.13), business.md(-11.30)
------------------------------------------------------------------------------


Q: What is the CEO's home address?
A: The CEO's home address is not provided in the given context.
   sources: shipping.md(-11.40), refunds.md(-11.40), business.md(-11.41)
------------------------------------------------------------------------------


Q: How long does Northwind Supply take to process a refund?
A: Northwind Supply processes refunds within 5 business days after receiving the proof of purchase and the order number.
   sources: refunds.md(-0.52), refunds.md(-4.06), privacy.md(-8.81)
------------------------------------------------------------------------------


---
## Step 11 — Memory: making it a conversation

A query engine answers one question and forgets. Ask *"and for international orders?"* on its
own and it has lost the subject.

`chat_mode="condense_plus_context"` fixes this in two steps:

1. rewrite the follow-up into a standalone question using the chat history;
2. retrieve using that *rewritten* question.

So memory improves **retrieval**, not just fluency. Keep `token_limit` modest: chat history and
retrieved chunks share the same 2048-token window.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Makes the assistant remember the conversation, so follow-ups like 'and for international?' make sense.
# WHAT THIS CELL DOES : Turns the query engine into a chatbot that remembers the conversation,
#                       then prints what it remembered.
# INPUT               : Three questions in a row. The 2nd and 3rd ("And for international orders?")
#                       only make sense if the bot remembers what was said before.
# OUTPUT              : USER / BOT lines for each turn, then the stored memory (who said what).
#                       Small models can lose track of the topic - check each reply against the question.
# =============================================================================
from llama_index.core.memory import Memory   # WHY: stores the conversation so far

# token_limit caps how much history is kept (history and documents share the 2048-token window).
memory = Memory.from_defaults(session_id="workshop-user", token_limit=1200)

# "condense_plus_context" = (1) rewrite the follow-up as a stand-alone question using the history,
#                           (2) search with that rewritten question, (3) answer from the found text.
chat = index.as_chat_engine(
    chat_mode="condense_plus_context",
    memory=memory,
    similarity_top_k=4,              # fetch 4 chunks...
    node_postprocessors=[reranker],  # ...then rerank (keeps the best 3)
    system_prompt=(                  # standing instructions the model receives on every turn
        "You are a customer support assistant for Northwind Supply. "
        "Answer only from the retrieved policy text. Be brief. "
        "If the policy does not cover it, say you don't know."
    ),
)

# Simulate a conversation.
for turn in [
    "How long do refunds take?",
    "And for international orders?",          # follow-up: relies on memory of the previous question
    "What about the shipping time for those?",
]:
    print("USER :", turn)
    print("BOT  :", str(chat.chat(turn)).strip())
    print("-" * 78)

# Show what is stored in memory: one line per message (speaker, then the first 110 characters).
print("\nMEMORY CONTENTS")
for message in memory.get():
    print(f"{str(message.role):<20} {' '.join(str(message.content).split())[:110]}")

USER : How long do refunds take?


BOT  : Refunds are issued to the original payment method within 5 business days. For international orders, refunds take 10 business days due to cross-border settlement times between the acquiring and issuing banks.
------------------------------------------------------------------------------
USER : And for international orders?


BOT  : For international orders, refunds take 10 business days because of cross-border settlement times between the acquiring and issuing banks.
------------------------------------------------------------------------------
USER : What about the shipping time for those?


BOT  : The shipping time for domestic deliveries is 3 to 5 business days, while for international deliveries, it can range from 7 to 14 business days.
------------------------------------------------------------------------------

MEMORY CONTENTS
MessageRole.USER     How long do refunds take?
MessageRole.ASSISTANT Refunds are issued to the original payment method within 5 business days. For international orders, refunds ta
MessageRole.USER     And for international orders?
MessageRole.ASSISTANT For international orders, refunds take 10 business days because of cross-border settlement times between the a
MessageRole.USER     What about the shipping time for those?
MessageRole.ASSISTANT The shipping time for domestic deliveries is 3 to 5 business days, while for international deliveries, it can 


---
## Step 12 — A tiny evaluation set

"It felt better" is not a measurement. A list of questions with known-correct sources turns
tuning from an argument into an experiment.

We measure **hit rate**: how often the correct document appears in the retrieved results.
This is a *retrieval* metric — it says nothing about the wording of the answer, which is exactly
why it is useful. Retrieval and generation should be measured separately.

In [ ]:
# =============================================================================
# IN PLAIN WORDS      : Gives the search a quick exam: 10 questions where we know the right document. Shows the score.
# WHAT THIS CELL DOES : Measures how good our search is, using test questions for which we already
#                       know the correct document.
# INPUT               : 10 test questions, each paired with the file that should be found (EVAL list).
# OUTPUT              : A "hit rate" for two search methods = the share of questions whose correct
#                       document appears in the top 3 results (100% = every question found its document).
#                       Each failed question is listed as a MISS.
# =============================================================================
# (question, file that contains the answer)
EVAL = [
    ("How long do refunds take?",                 "refunds.md"),
    ("Can I return something after six weeks?",   "refunds.md"),
    ("When will my parcel arrive?",               "shipping.md"),
    ("Do you ship abroad?",                       "shipping.md"),
    ("Is water damage covered?",                  "warranty.md"),
    ("How do I start a warranty claim?",          "warranty.md"),
    ("I forgot my password",                      "accounts.md"),
    ("How long is my data kept?",                 "privacy.md"),
    ("Do you sell my information?",               "privacy.md"),
    ("What is the bulk pricing for 100 users?",   "business.md"),
]

# Search method 1: plain vector search, keep the top 3.
def plain_search(question):
    return index.as_retriever(similarity_top_k=3).retrieve(question)

# Search method 2: fetch 10 candidates, then rerank and keep the best 3.
def reranked_search(question):
    candidates = index.as_retriever(similarity_top_k=10).retrieve(question)
    return reranker.postprocess_nodes(candidates, QueryBundle(question))

# Runs a search method on every test question and returns the share where the right file was found.
#   input : a search function
#   output: hit rate between 0.0 and 1.0 (and prints every miss)
def hit_rate(search):
    hits = 0
    for question, expected_file in EVAL:
        found_files = [n.metadata["source"] for n in search(question)]
        if expected_file in found_files:
            hits += 1
        else:
            print(f"    MISS  {question!r}  expected {expected_file}, got {found_files}")
    return hits / len(EVAL)

print(f"vector top-3                 hit rate = {hit_rate(plain_search):.0%}")
print(f"top-10 then reranked to 3    hit rate = {hit_rate(reranked_search):.0%}")
print("\nAdd your own questions to EVAL and re-run before changing anything else.")

vector top-3                 hit rate = 100%


top-10 then reranked to 3    hit rate = 100%

Add your own questions to EVAL and re-run before changing anything else.


### 🧪 EXPERIMENT 3 — make a change, then measure it

Pick one change and measure its effect on the hit rate above. Change **one thing at a time**.

* Set `chunk_size=96` in Step 4, then re-run from Step 4 down to this cell.
* Drop `similarity_top_k` to 1 in `plain_search` and see which questions start failing.
* Swap the embedding model in Step 2 to `BAAI/bge-base-en-v1.5` (larger, slower, usually better),
  then re-run from Step 2 down.
* Add the hybrid retriever from Step 8 as a third search method.

---
## Where to take this next

**Use your own documents.** Put PDFs or Markdown files in the `data` folder and re-run from
Step 3 (remember to give each file a category in `CATEGORY`). `SimpleDirectoryReader` also reads DOCX and TXT.

**Try a bigger model.** Change `LLM_MODEL` to `Qwen/Qwen3-1.7B` or `meta-llama/Llama-3.2-1B-Instruct`
(the Llama licence must be accepted on Hugging Face first). Raise `context_window` to 4096 and you can afford more chunks.

**Add real evaluation.** LlamaIndex ships `FaithfulnessEvaluator` (is the answer supported by the retrieved text?)
and `RelevancyEvaluator` (did it address the question?) to sit alongside the hit rate above.

---

### The three things worth remembering

1. **A small model is a good reader and a poor encyclopaedia.** Do not ask it what it knows —
   give it what it needs to read.
2. **Retrieval quality is the ceiling on answer quality.** Debug the retriever before the prompt.
3. **Chunking and reranking beat model size.** Time spent there pays back more than shopping for a bigger model.